# 🏥 Medical AI Pipeline — Unified System

```
User Input
    │
    ▼
[1] Spell Corrector       T5-small → fixes typos + expands medical abbreviations
    │
    ▼
[2] Intent Classifier     TF-IDF + LR → Medical / Non-Medical routing
    ├── Non-Medical ──▶ Polite deflection
    └── Medical ─────▶
                │
                ▼
           [3] Disease Predictor    DistilBERT → Top-3 diseases + confidence
                │
                ▼
           [4] Drug Lookup          MED-RT API → Drugs per predicted disease
                │
                ▼
           [5] Interaction Checker  CSV database → Drug-drug interaction warnings
                │
                ▼
           [6] Structured Response
```



## ── SECTION 0: Environment & Seeds ──

In [2]:
import sys, platform, importlib, datetime

print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Date/Time : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.platform()}")
for pkg in ["torch", "transformers", "sklearn", "joblib", "pandas", "numpy", "requests", "difflib"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:<20} {getattr(m, '__version__', 'stdlib')}")
    except ImportError:
        print(f"  {pkg:<20} NOT INSTALLED")
print("=" * 60)

ENVIRONMENT INFO
Date/Time : 2026-04-06 10:48:13
Python    : 3.13.9
Platform  : Windows-11-10.0.26100-SP0
  torch                2.10.0+cpu
  transformers         5.0.0
  sklearn              1.7.2
  joblib               1.5.2
  pandas               2.3.3
  numpy                2.3.5
  requests             2.32.5
  difflib              stdlib


In [1]:
import random, os, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Seeds set (SEED={SEED}). Device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print(f"Seeds set (SEED={SEED}). PyTorch not found — CPU mode.")

Seeds set (SEED=42). Device: cpu


In [3]:
import logging

LOG_FILE = "medical_pipeline.log"
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("MedicalPipeline")
logger.info("Medical AI Pipeline started")
print(f"Logging → {LOG_FILE}")

2026-04-06 10:48:31,520 | INFO     | Medical AI Pipeline started
Logging → medical_pipeline.log


## ── SECTION 1: Load Model [1] — Spell Corrector (T5-small) ──

Fixes typos and expands medical abbreviations using a fine-tuned T5-small model.

In [4]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

# ── CONFIG  ──────────────────────
SPELL_MODEL_DIR = "./medical_spellcheck_final"   # saved by Spell_Corrector_Fixed.ipynb
SPELL_PREFIX    = "fix: "
SPELL_MAX_LEN   = 64
# ─────────────────────────────────────────────────────────────────────────

logger.info(f"Loading Spell Corrector from '{SPELL_MODEL_DIR}' ...")

try:
    _spell_tokenizer = AutoTokenizer.from_pretrained(SPELL_MODEL_DIR)
    _spell_model     = T5ForConditionalGeneration.from_pretrained(SPELL_MODEL_DIR).to(DEVICE)
    _spell_model.eval()
    print(f"✅ [1] Spell Corrector loaded  ({sum(p.numel() for p in _spell_model.parameters()):,} params)")
    logger.info("Spell Corrector loaded OK")
    SPELL_AVAILABLE = True
except Exception as e:
    logger.warning(f"Spell Corrector load failed: {e}")
    print(f"⚠️  [1] Spell Corrector not found at '{SPELL_MODEL_DIR}'.")
    print("    Running without spell correction (pass-through mode).")
    SPELL_AVAILABLE = False


def spell_correct(text: str) -> str:
    """
    Stage 1 — Spell Corrector.
    Uses fine-tuned T5-small to fix typos and expand medical abbreviations.
    Falls back to identity if model is not loaded.
    """
    if not SPELL_AVAILABLE or not isinstance(text, str) or not text.strip():
        return text
    try:
        inputs = _spell_tokenizer(
            SPELL_PREFIX + text,
            return_tensors="pt",
            max_length=SPELL_MAX_LEN,
            truncation=True,
        ).to(DEVICE)
        with torch.no_grad():
            out = _spell_model.generate(
                **inputs,
                max_length=SPELL_MAX_LEN,
                num_beams=4,
                no_repeat_ngram_size=2,
                early_stopping=True,
            )
        return _spell_tokenizer.decode(out[0], skip_special_tokens=True)
    except Exception as e:
        logger.warning(f"Spell correction failed for '{text[:50]}': {e}")
        return text   # safe fallback


# Quick smoke test
print("\nSmoke test:")
test_in = "pt takng pcm for fevr"
test_out = spell_correct(test_in)
print(f"  Input : {test_in}")
print(f"  Output: {test_out}")

2026-04-06 10:48:40,498 | INFO     | Loading Spell Corrector from './medical_spellcheck_final' ...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ [1] Spell Corrector loaded  (60,506,624 params)
2026-04-06 10:48:41,150 | INFO     | Spell Corrector loaded OK

Smoke test:
  Input : pt takng pcm for fevr
  Output: patient taking paracetamol for fever


## ── SECTION 2: Load Model [2] — Intent Classifier (TF-IDF + LR) ──

Routes corrected text to Medical or Non-Medical path.

In [5]:
import re, string, unicodedata, joblib

# ── CONFIG — update paths if needed ──────────────────────────────────────
INTENT_MODEL_PATH    = "intent_classifier_medical_safe.pkl"    # saved by intent_classifier_fixed.ipynb
INTENT_METADATA_PATH = "intent_classifier_metadata.json"       # saved alongside model
# ─────────────────────────────────────────────────────────────────────────

# ── Text cleaning (must match what was used during training) ─────────────
_CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'t": " not", "'ve": " have", "'m": " am",
}

def _clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.lower()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    for k, v in _CONTRACTIONS.items():
        text = text.replace(k, v)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


logger.info(f"Loading Intent Classifier from '{INTENT_MODEL_PATH}' ...")

try:
    _intent_pipeline     = joblib.load(INTENT_MODEL_PATH)
    import json
    with open(INTENT_METADATA_PATH) as f:
        _intent_meta = json.load(f)
    INTENT_THRESHOLD = _intent_meta.get("optimal_threshold", 0.5)
    print(f"✅ [2] Intent Classifier loaded  (threshold={INTENT_THRESHOLD:.4f})")
    logger.info(f"Intent Classifier loaded OK  threshold={INTENT_THRESHOLD}")
    INTENT_AVAILABLE = True
except Exception as e:
    logger.warning(f"Intent Classifier load failed: {e}")
    print(f"⚠️  [2] Intent Classifier not found at '{INTENT_MODEL_PATH}'.")
    print("    All inputs will be treated as Medical (pass-through).")
    INTENT_AVAILABLE  = False
    INTENT_THRESHOLD  = 0.5


def classify_intent(text: str) -> dict:
    """
    Stage 2 — Intent Classifier.
    Returns {label, medical_prob, threshold_used}.
    label = 'Medical' | 'Non-Medical'
    """
    if not INTENT_AVAILABLE:
        return {"label": "Medical", "medical_prob": 1.0, "threshold_used": INTENT_THRESHOLD}
    cleaned      = _clean_text(text)
    proba        = _intent_pipeline.predict_proba([cleaned])[0]
    medical_prob = proba[list(_intent_pipeline.classes_).index(1)]
    label        = "Medical" if medical_prob >= INTENT_THRESHOLD else "Non-Medical"
    return {"label": label, "medical_prob": round(medical_prob, 4), "threshold_used": INTENT_THRESHOLD}


# Quick smoke test
print("\nSmoke test:")
for t in ["I have chest pain and fever", "Tell me a joke"]:
    r = classify_intent(t)
    print(f"  '{t}' → {r['label']}  (P={r['medical_prob']})")

2026-04-06 10:48:46,947 | INFO     | Loading Intent Classifier from 'intent_classifier_medical_safe.pkl' ...
✅ [2] Intent Classifier loaded  (threshold=0.6187)
2026-04-06 10:48:47,243 | INFO     | Intent Classifier loaded OK  threshold=0.6187

Smoke test:
  'I have chest pain and fever' → Medical  (P=0.9994)
  'Tell me a joke' → Non-Medical  (P=0.0447)


## ── SECTION 3: Load Model [3] — Disease Predictor (DistilBERT) ──

Predicts Top-3 diseases with confidence scores.

In [6]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

# ── CONFIG — update path if needed ───────────────────────────────────────
DISEASE_MODEL_DIR = "./zolex_disease_model_v2"   # saved by disease_prediction_improved.ipynb
DISEASE_MAX_LEN   = 128
DISEASE_TOP_K     = 3
# ─────────────────────────────────────────────────────────────────────────

logger.info(f"Loading Disease Predictor from '{DISEASE_MODEL_DIR}' ...")

try:
    _disease_tokenizer = DistilBertTokenizerFast.from_pretrained(DISEASE_MODEL_DIR)
    _disease_model     = DistilBertForSequenceClassification.from_pretrained(DISEASE_MODEL_DIR)
    _disease_model.to(DEVICE)
    _disease_model.eval()
    print(f"✅ [3] Disease Predictor loaded  ({_disease_model.config.num_labels} classes)")
    logger.info(f"Disease Predictor loaded OK — {_disease_model.config.num_labels} classes")
    DISEASE_AVAILABLE = True
except Exception as e:
    logger.warning(f"Disease Predictor load failed: {e}")
    print(f"⚠️  [3] Disease Predictor not found at '{DISEASE_MODEL_DIR}'.")
    print("    Disease prediction will return a placeholder.")
    DISEASE_AVAILABLE = False


def predict_disease(text: str, top_k: int = DISEASE_TOP_K) -> list:
    """
    Stage 3 — Disease Predictor.
    Returns list of {disease, confidence} dicts, highest confidence first.
    """
    if not DISEASE_AVAILABLE:
        return [{"disease": "Unknown (model not loaded)", "confidence": 0.0}]
    inputs = _disease_tokenizer(
        text, return_tensors="pt", truncation=True,
        padding=True, max_length=DISEASE_MAX_LEN,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = _disease_model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]
    return [
        {"disease": _disease_model.config.id2label[int(i)], "confidence": round(float(probs[i]), 4)}
        for i in top_idx
    ]


# Quick smoke test
print("\nSmoke test:")
results = predict_disease("patient has fever cough and difficulty breathing")
for r in results:
    print(f"  {r['disease']:<50} {r['confidence']:.4f}")

2026-04-06 10:48:51,385 | INFO     | Loading Disease Predictor from './zolex_disease_model_v2' ...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ [3] Disease Predictor loaded  (607 classes)
2026-04-06 10:48:51,775 | INFO     | Disease Predictor loaded OK — 607 classes

Smoke test:
  asthma                                             0.2841
  acute bronchospasm                                 0.2003
  acute respiratory distress syndrome (ards)         0.1146


## ── SECTION 4: Load Model [4] — Drug Lookup (MED-RT API) ──

Fetches indicated drugs per disease from the NLM MED-RT / RxNav API with caching.

In [7]:
import requests, time, json, csv, functools
from typing import Optional, List, Dict, Any

MEDRT_BASE_URL   = "https://rxnav.nlm.nih.gov/REST"
MEDRT_TIMEOUT    = 10
MEDRT_MAX_RETRY  = 3
MEDRT_RETRY_WAIT = 1.5
MEDRT_CACHE_FILE = "medrt_cache.json"
DRUG_LIMIT       = 10    # drugs to retrieve per disease (None = all)


class MedRTClient:
    """MED-RT API client with retry, caching, and structured error handling."""

    def __init__(self, use_disk_cache: bool = True):
        self._cache: Dict[str, dict] = {}
        self.use_disk_cache = use_disk_cache
        if use_disk_cache and os.path.exists(MEDRT_CACHE_FILE):
            try:
                with open(MEDRT_CACHE_FILE, encoding="utf-8") as f:
                    self._cache = json.load(f)
                logger.info(f"MED-RT disk cache loaded: {len(self._cache)} entries")
            except Exception as e:
                logger.warning(f"Could not load MED-RT cache: {e}")

    def _get(self, url: str, params: dict) -> Optional[dict]:
        for attempt in range(1, MEDRT_MAX_RETRY + 1):
            try:
                r = requests.get(url, params=params, timeout=MEDRT_TIMEOUT)
                r.raise_for_status()
                return r.json()
            except requests.exceptions.Timeout:
                logger.warning(f"MED-RT timeout (attempt {attempt})")
            except requests.exceptions.ConnectionError as e:
                logger.warning(f"MED-RT connection error: {e}")
            except requests.exceptions.HTTPError as e:
                logger.error(f"MED-RT HTTP error: {e}")
                return None
            except ValueError:
                return None
            if attempt < MEDRT_MAX_RETRY:
                time.sleep(MEDRT_RETRY_WAIT * attempt)
        return None

    def _find_class_id(self, name: str) -> Optional[dict]:
        data = self._get(f"{MEDRT_BASE_URL}/rxclass/class/byName.json",
                         {"className": name, "relaSource": "MEDRT"})
        if not data:
            return None
        concepts = data.get("rxclassMinConceptList", {}).get("rxclassMinConcept", [])
        return {"classId": concepts[0]["classId"], "className": concepts[0]["className"]} if concepts else None

    def _get_members(self, class_id: str) -> List[str]:
        data = self._get(f"{MEDRT_BASE_URL}/rxclass/classMembers.json",
                         {"classId": class_id, "relaSource": "MEDRT", "rela": "may_treat", "trans": "0"})
        if not data:
            return []
        members = data.get("drugMemberGroup", {}).get("drugMember", [])
        return sorted({m["minConcept"]["name"] for m in members})

    def get_drugs_for_disease(self, disease: str, limit: Optional[int] = DRUG_LIMIT) -> dict:
        key = disease.strip().lower()
        if key in self._cache:
            cached = self._cache[key]
            drugs  = cached["drugs"]
            shown  = drugs[:limit] if limit else drugs
            return {"disease_query": disease, "matched_class": cached.get("matched_class"),
                    "class_id": cached.get("class_id"), "drugs": shown,
                    "total_found": len(drugs), "returned": len(shown), "cached": True}

        concept = self._find_class_id(disease)
        if not concept:
            return {"disease_query": disease, "matched_class": None, "class_id": None,
                    "drugs": [], "total_found": 0, "returned": 0, "cached": False}

        all_drugs = self._get_members(concept["classId"])
        self._cache[key] = {"drugs": all_drugs, "matched_class": concept["className"],
                            "class_id": concept["classId"]}
        if self.use_disk_cache:
            try:
                with open(MEDRT_CACHE_FILE, "w", encoding="utf-8") as f:
                    json.dump(self._cache, f, indent=2)
            except Exception:
                pass

        shown = all_drugs[:limit] if limit else all_drugs
        return {"disease_query": disease, "matched_class": concept["className"],
                "class_id": concept["classId"], "drugs": shown,
                "total_found": len(all_drugs), "returned": len(shown), "cached": False}


_drug_client = MedRTClient(use_disk_cache=True)
print("✅ [4] Drug Lookup client ready  (MED-RT API + disk cache)")
logger.info("MedRTClient initialised")


def lookup_drugs(disease: str, limit: int = DRUG_LIMIT) -> list:
    """
    Stage 4 — Drug Lookup.
    Returns list of drug name strings for a given disease.
    """
    result = _drug_client.get_drugs_for_disease(disease, limit=limit)
    return result.get("drugs", [])


# Quick smoke test
print("\nSmoke test (Hypertension):")
drugs = lookup_drugs("Hypertension", limit=5)
print(f"  First 5 drugs: {drugs}")

2026-04-06 10:48:55,429 | INFO     | MED-RT disk cache loaded: 23 entries
✅ [4] Drug Lookup client ready  (MED-RT API + disk cache)
2026-04-06 10:48:55,431 | INFO     | MedRTClient initialised

Smoke test (Hypertension):
  First 5 drugs: ['Angelica sinensis preparation', 'acebutolol', 'acebutolol hydrochloride', 'aliskiren', 'aliskiren hemifumarate']


## ── SECTION 5: Load Model [5] — Interaction Checker (Rule-based + CSV) ──

Checks drug-drug interactions against a CSV database.

In [8]:
import pandas as pd
import difflib
from itertools import combinations

# ── CONFIG — update CSV path if needed ───────────────────────────────────
INTERACTION_CSV = "db_drug_interactions.csv"   # required by check_interactions_fixed.ipynb
FUZZY_THRESHOLD = 0.82
# ─────────────────────────────────────────────────────────────────────────

# Synonym map (brand → generic)
_DRUG_SYNONYMS = {
    "acetaminophen": "paracetamol", "tylenol": "paracetamol", "panadol": "paracetamol",
    "advil": "ibuprofen", "motrin": "ibuprofen", "nurofen": "ibuprofen",
    "aspirin": "acetylsalicylic acid", "asa": "acetylsalicylic acid",
    "lasix": "furosemide", "zoloft": "sertraline", "prozac": "fluoxetine",
    "plavix": "clopidogrel", "coumadin": "warfarin", "glucophage": "metformin",
    "synthroid": "levothyroxine", "lipitor": "atorvastatin", "zocor": "simvastatin",
    "nexium": "esomeprazole", "prilosec": "omeprazole", "xanax": "alprazolam",
    "valium": "diazepam", "amoxil": "amoxicillin", "zithromax": "azithromycin",
    "norvasc": "amlodipine", "ventolin": "albuterol", "aleve": "naproxen",
}
_HIGH_KW   = frozenset(["severe", "avoid", "major", "contraindicated", "life-threatening",
                        "fatal", "death", "serious", "do not", "potentially fatal"])
_MEDIUM_KW = frozenset(["moderate", "caution", "monitor", "increased risk",
                        "may increase", "reduced", "decreased", "adjust"])


class DrugInteractionChecker:
    """Rule-based + CSV interaction checker with fuzzy synonym matching."""

    REQUIRED_COLS = {"drug 1", "drug 2", "interaction description"}

    def __init__(self, csv_path: str, fuzzy_threshold: float = FUZZY_THRESHOLD,
                 synonyms: dict = None):
        self.fuzzy_threshold = fuzzy_threshold
        self.synonyms = {k.lower(): v.lower() for k, v in (synonyms or _DRUG_SYNONYMS).items()}
        self.db: Dict[str, tuple] = {}
        self.all_drug_names: List[str] = []
        self._load(csv_path)

    def _load(self, path: str) -> None:
        df = pd.read_csv(path, low_memory=False)
        df.columns = df.columns.str.strip()
        col_map = {c.lower(): c for c in df.columns}
        missing = self.REQUIRED_COLS - set(col_map.keys())
        if missing:
            raise ValueError(f"CSV missing columns: {missing}")
        d1, d2, desc = col_map["drug 1"], col_map["drug 2"], col_map["interaction description"]
        sev_col = col_map.get("severity")
        df = df.dropna(subset=[d1, d2]).copy()
        df[d1]   = df[d1].astype(str).str.strip().str.lower()
        df[d2]   = df[d2].astype(str).str.strip().str.lower()
        df[desc] = df[desc].astype(str).str.strip()
        cols = [d1, d2, desc] + ([sev_col] if sev_col else [])
        for row in df[cols].itertuples(index=False):
            r1, r2, rdesc = row[0], row[1], row[2]
            sev_raw = row[3] if sev_col else ""
            key = "_".join(sorted([r1, r2]))
            if key not in self.db:
                self.db[key] = (self._severity(sev_raw, rdesc), rdesc)
        all_drugs: set = set()
        for k in self.db:
            all_drugs.update(k.split("_", 1))
        self.all_drug_names = sorted(all_drugs)
        logger.info(f"Interaction DB loaded: {len(self.db):,} pairs")

    @staticmethod
    def _severity(sev_raw: str, desc: str) -> str:
        s = sev_raw.strip().lower()
        if any(x in s for x in ["high", "major", "severe", "contraindicated"]):
            return "HIGH"
        if any(x in s for x in ["moderate", "medium"]):
            return "MEDIUM"
        if any(x in s for x in ["low", "minor"]):
            return "LOW"
        dl = desc.lower()
        if any(kw in dl for kw in _HIGH_KW):
            return "HIGH"
        if any(kw in dl for kw in _MEDIUM_KW):
            return "MEDIUM"
        return "LOW"

    def _normalise(self, name: str) -> str:
        c = name.strip().lower()
        if c in self.synonyms:
            return self.synonyms[c]
        if c in set(self.all_drug_names):
            return c
        matches = difflib.get_close_matches(c, self.all_drug_names, n=1, cutoff=self.fuzzy_threshold)
        return matches[0] if matches else c

    def check_risk(self, drug_list: list) -> list:
        if len(drug_list) < 2:
            return []
        norm = [self._normalise(d) for d in drug_list]
        seen, warnings = set(), []
        for d1, d2 in combinations(norm, 2):
            key = "_".join(sorted([d1, d2]))
            if key in seen:
                continue
            seen.add(key)
            if key in self.db:
                sev, desc = self.db[key]
                warnings.append({"Pair": f"{d1.title()} ⟷ {d2.title()}",
                                  "Drug1": d1, "Drug2": d2, "Risk": sev, "Details": desc})
        warnings.sort(key=lambda x: {"HIGH": 0, "MEDIUM": 1, "LOW": 2}.get(x["Risk"], 9))
        return warnings


logger.info(f"Loading Interaction Checker from '{INTERACTION_CSV}' ...")
try:
    _interaction_checker = DrugInteractionChecker(INTERACTION_CSV)
    print(f"✅ [5] Interaction Checker loaded  ({len(_interaction_checker.db):,} interaction pairs)")
    INTERACTION_AVAILABLE = True
except FileNotFoundError:
    print(f"⚠️  [5] Interaction Checker: '{INTERACTION_CSV}' not found.")
    print("    Drug interaction warnings will be skipped.")
    INTERACTION_AVAILABLE = False
    _interaction_checker  = None
except Exception as e:
    logger.warning(f"Interaction Checker load failed: {e}")
    print(f"⚠️  [5] Interaction Checker failed: {e}")
    INTERACTION_AVAILABLE = False
    _interaction_checker  = None


def check_interactions(drug_list: list) -> list:
    """
    Stage 5 — Interaction Checker.
    Returns list of {Pair, Risk, Details} dicts sorted HIGH → MEDIUM → LOW.
    """
    if not INTERACTION_AVAILABLE or len(drug_list) < 2:
        return []
    return _interaction_checker.check_risk(drug_list)


# Quick smoke test
print("\nSmoke test (warfarin + aspirin):")
result = check_interactions(["warfarin", "aspirin"])
if result:
    print(f"  [{result[0]['Risk']}] {result[0]['Pair']} → {result[0]['Details'][:60]}...")
else:
    print("  No interactions found (or checker not loaded).")

2026-04-06 10:49:02,532 | INFO     | Loading Interaction Checker from 'db_drug_interactions.csv' ...
2026-04-06 10:49:06,089 | INFO     | Interaction DB loaded: 191,135 pairs
✅ [5] Interaction Checker loaded  (191,135 interaction pairs)

Smoke test (warfarin + aspirin):
  [MEDIUM] Warfarin ⟷ Acetylsalicylic Acid → Warfarin may increase the anticoagulant activities of Acetyl...


## ── SECTION 6: Pipeline Status Summary ──

In [9]:
print("\n" + "=" * 60)
print("MEDICAL AI PIPELINE — MODEL STATUS")
print("=" * 60)

STATUS = [
    ("[1] Spell Corrector",    "T5-small",           SPELL_AVAILABLE),
    ("[2] Intent Classifier",  "TF-IDF + LR",        INTENT_AVAILABLE),
    ("[3] Disease Predictor",  "DistilBERT",          DISEASE_AVAILABLE),
    ("[4] Drug Lookup",        "MED-RT API",          True),            
    ("[5] Interaction Checker","CSV rule-based",      INTERACTION_AVAILABLE),
]

for name, model_type, ok in STATUS:
    icon = "✅" if ok else "⚠️ "
    note = "READY" if ok else "PASS-THROUGH"
    print(f"  {icon} {name:<28} {model_type:<20} {note}")

print("=" * 60)
all_ready = all(ok for _, _, ok in STATUS)
print(f"\nPipeline status: {'🟢 FULLY OPERATIONAL' if all_ready else '🟡 PARTIAL (some models not loaded)'}")


MEDICAL AI PIPELINE — MODEL STATUS
  ✅ [1] Spell Corrector          T5-small             READY
  ✅ [2] Intent Classifier        TF-IDF + LR          READY
  ✅ [3] Disease Predictor        DistilBERT           READY
  ✅ [4] Drug Lookup              MED-RT API           READY
  ✅ [5] Interaction Checker      CSV rule-based       READY

Pipeline status: 🟢 FULLY OPERATIONAL


## ── SECTION 7: Unified End-to-End Pipeline Function ──

In [10]:
def run_pipeline(
    user_input: str,
    top_k_diseases: int = 3,
    drugs_per_disease: int = DRUG_LIMIT,
    verbose: bool = True,
) -> dict:
    """
    End-to-end Medical AI Pipeline.

    Stages:
        1. Spell Corrector  — T5-small
        2. Intent Classifier — TF-IDF + LR (routes Non-Medical to polite deflection)
        3. Disease Predictor — DistilBERT (Top-K diseases)
        4. Drug Lookup      — MED-RT API
        5. Interaction Checker — CSV rule-based
        6. Structured Response

    Returns a structured dict with all intermediate + final results.
    """
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input must be a non-empty string")

    output = {
        "original_input":   user_input,
        "corrected_input":  None,
        "intent":           None,
        "diseases":         [],
        "drugs_by_disease": {},
        "all_drugs":        [],
        "interactions":     [],
        "response":         "",
        "routed_to":        None,
    }

    sep = "─" * 55
    if verbose:
        print(sep)
        print(f"  INPUT: {user_input}")
        print(sep)

    # ── Stage 1: Spell Correction ────────────────────────────
    corrected = spell_correct(user_input)
    output["corrected_input"] = corrected
    if verbose:
        label = "(unchanged)" if corrected == user_input else f"→ '{corrected}'"
        print(f"  [1] Spell Corrector  : {label}")

    # ── Stage 2: Intent Classification ──────────────────────
    intent = classify_intent(corrected)
    output["intent"] = intent
    if verbose:
        print(f"  [2] Intent Classifier: {intent['label']}  "
              f"(P(medical)={intent['medical_prob']}, threshold={intent['threshold_used']})")

    # ── Non-Medical routing ──────────────────────────────────
    if intent["label"] == "Non-Medical":
        output["routed_to"] = "Non-Medical"
        msg = ("I'm a medical assistant and can only help with health-related "
               "questions. Please describe your symptoms or medical concern.")
        output["response"] = msg
        if verbose:
            print(f"  ↳ Routed to: Polite Deflection")
            print(f"\n  RESPONSE: {msg}")
        return output

    output["routed_to"] = "Medical"

    # ── Stage 3: Disease Prediction ──────────────────────────
    diseases = predict_disease(corrected, top_k=top_k_diseases)
    output["diseases"] = diseases
    if verbose:
        print(f"  [3] Disease Predictor:")
        for i, d in enumerate(diseases, 1):
            print(f"       {i}. {d['disease']:<45} {d['confidence']*100:.1f}%")

    # ── Stage 4: Drug Lookup ─────────────────────────────────
    all_drugs_flat: List[str] = []
    if verbose:
        print(f"  [4] Drug Lookup (MED-RT):")
    for d in diseases:
        disease_name = d["disease"]
        drugs = lookup_drugs(disease_name, limit=drugs_per_disease)
        output["drugs_by_disease"][disease_name] = drugs
        all_drugs_flat.extend(drugs)
        if verbose:
            short_list = ", ".join(drugs[:5]) + (" ..." if len(drugs) > 5 else "")
            print(f"       {disease_name:<35}: {short_list if drugs else '(no results)'}")

    # Deduplicate while preserving order
    seen_drugs: set = set()
    unique_drugs: List[str] = []
    for drug in all_drugs_flat:
        if drug.lower() not in seen_drugs:
            seen_drugs.add(drug.lower())
            unique_drugs.append(drug)
    output["all_drugs"] = unique_drugs

    # ── Stage 5: Interaction Checking ────────────────────────
    interactions = check_interactions(unique_drugs)
    output["interactions"] = interactions
    if verbose:
        print(f"  [5] Interaction Checker: {len(interactions)} interaction(s) found")
        icons = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}
        for ix in interactions:
            print(f"       {icons.get(ix['Risk'], '⚪')} [{ix['Risk']}] {ix['Pair']}")
            print(f"          → {ix['Details'][:80]}")

    # ── Stage 6: Structured Response ─────────────────────────
    lines = []
    lines.append("📋 MEDICAL ASSESSMENT")
    lines.append(f"Input (corrected): {corrected}")
    lines.append("")
    lines.append("🔍 Possible Conditions (confidence):")
    for d in diseases:
        lines.append(f"  • {d['disease']} ({d['confidence']*100:.1f}%)")
    lines.append("")
    lines.append("💊 Related Medications:")
    for disease_name, drugs in output["drugs_by_disease"].items():
        if drugs:
            lines.append(f"  {disease_name}: {', '.join(drugs[:5])}"
                         + (f" (+{len(drugs)-5} more)" if len(drugs) > 5 else ""))
        else:
            lines.append(f"  {disease_name}: (no medications found in MED-RT)")
    if interactions:
        lines.append("")
        lines.append("⚠️ Drug Interaction Warnings:")
        for ix in interactions:
            lines.append(f"  [{ix['Risk']}] {ix['Pair']}: {ix['Details'][:100]}")
    lines.append("")
    lines.append("⚕️ Note: This is an AI-assisted analysis. Always consult a licensed physician.")

    output["response"] = "\n".join(lines)

    if verbose:
        print()
        print(sep)
        print(output["response"])
        print(sep)

    logger.info(f"Pipeline complete | intent={intent['label']} | "
                f"diseases={[d['disease'] for d in diseases]} | "
                f"drugs={len(unique_drugs)} | interactions={len(interactions)}")
    return output


print("✅ run_pipeline() function defined — ready to use.")

✅ run_pipeline() function defined — ready to use.


## ── SECTION 8: Test Runs ──

In [11]:
# ── Test 1: Medical query with typos + abbreviations ──────────────────────
result1 = run_pipeline("pt has hgh fevr and servere cough", verbose=True)

───────────────────────────────────────────────────────
  INPUT: pt has hgh fevr and servere cough
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'patient has fever and server cough'
  [2] Intent Classifier: Medical  (P(medical)=0.9989, threshold=0.6187)
  [3] Disease Predictor:
       1. conjunctivitis due to bacteria                10.5%
       2. abscess of nose                               8.0%
       3. abscess of the pharynx                        5.9%
  [4] Drug Lookup (MED-RT):
       conjunctivitis due to bacteria     : (no results)
       abscess of nose                    : (no results)
       abscess of the pharynx             : (no results)
  [5] Interaction Checker: 0 interaction(s) found

───────────────────────────────────────────────────────
📋 MEDICAL ASSESSMENT
Input (corrected): patient has fever and server cough

🔍 Possible Conditions (confidence):
  • conjunctivitis due to bacteria (10.5%)
  • abscess of nose (8.0%)
  • abscess 

In [12]:
# ── Test 2: Non-medical query (should deflect politely) ────────────────────
result2 = run_pipeline("What is the weather like today?", verbose=True)

───────────────────────────────────────────────────────
  INPUT: What is the weather like today?
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'Wie sieht es heute aus?'
  [2] Intent Classifier: Non-Medical  (P(medical)=0.1285, threshold=0.6187)
  ↳ Routed to: Polite Deflection

  RESPONSE: I'm a medical assistant and can only help with health-related questions. Please describe your symptoms or medical concern.


In [13]:
# ── Test 3: Multi-symptom medical query ────────────────────────────────────
result3 = run_pipeline(
    "patient with chest pain shortness of breath and high blood pressure",
    verbose=True
)

───────────────────────────────────────────────────────
  INPUT: patient with chest pain shortness of breath and high blood pressure
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'patient with heart failure being given breath and high blood pressure'
  [2] Intent Classifier: Medical  (P(medical)=0.9837, threshold=0.6187)
  [3] Disease Predictor:
       1. sinus bradycardia                             34.7%
       2. paroxysmal ventricular tachycardia            20.4%
       3. sick sinus syndrome                           7.8%
  [4] Drug Lookup (MED-RT):
       sinus bradycardia                  : (no results)
       paroxysmal ventricular tachycardia : (no results)
       sick sinus syndrome                : (no results)
  [5] Interaction Checker: 0 interaction(s) found

───────────────────────────────────────────────────────
📋 MEDICAL ASSESSMENT
Input (corrected): patient with heart failure being given breath and high blood pressure

🔍 Possible Co

## ── SECTION 9: Batch Inference ──

In [15]:
BATCH_QUERIES = [
    "I have a severe headache and sensitivity to light",
    "pt takng pcm for fevr",
    "can you book me a flight?",
    "my child has rash itching and swollen lymph nodes",
    "hx of diabtes on metformin",
    "sudden joint pain and swelling in the knee",
    "how do I cook pasta?",
]

print("=" * 70)
print("BATCH RESULTS SUMMARY")
print("=" * 70)
print(f"{'#':<3} {'Input':<42} {'Intent':<12} {'Top Disease':<30} {'Warnings'}")
print("-" * 100)

for i, query in enumerate(BATCH_QUERIES, 1):
    try:
        r = run_pipeline(query, verbose=False)
        intent    = r["intent"]["label"]
        top_dis   = r["diseases"][0]["disease"] if r["diseases"] else "—"
        n_warn    = len(r["interactions"])
        warn_str  = f"{n_warn} 🔴" if n_warn > 0 else "none"
        print(f"  {i:<2} {query[:40]:<42} {intent:<12} {top_dis[:28]:<30} {warn_str}")
    except Exception as e:
        print(f"  {i:<2} {query[:40]:<42} ERROR: {e}")

print("=" * 70)

BATCH RESULTS SUMMARY
#   Input                                      Intent       Top Disease                    Warnings
----------------------------------------------------------------------------------------------------
2026-04-06 10:53:05,077 | INFO     | Pipeline complete | intent=Medical | diseases=['smoking or tobacco addiction', 'iridocyclitis', 'eustachian tube dysfunction (ear disorder)'] | drugs=3 | interactions=1
  1  I have a severe headache and sensitivity   Medical      smoking or tobacco addiction   1 🔴
2026-04-06 10:53:15,050 | INFO     | Pipeline complete | intent=Medical | diseases=['herpangina', 'noninfectious gastroenteritis', 'acute kidney injury'] | drugs=6 | interactions=6
  2  pt takng pcm for fevr                      Medical      herpangina                     6 🔴
  3  can you book me a flight?                  Non-Medical  —                              none
2026-04-06 10:53:24,792 | INFO     | Pipeline complete | intent=Medical | diseases=['herpangina', 'mo

## ── SECTION 10: Interactive Mode ──

> Uncomment the cell below to run an interactive session.

In [18]:
def run_interactive_pipeline():
    print("🏥 Medical AI Pipeline — Interactive Mode")
    print("   Describe your symptoms. Type 'quit' to exit.\n")
    while True:
        try:
            user_input = input("\n💬 Your symptoms: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break
        if user_input.lower() in ("quit", "exit", ""):
            print("Goodbye! 👋")
            break
        try:
            run_pipeline(user_input, verbose=True)
        except Exception as e:
            print(f"Error: {e}")

run_interactive_pipeline()

print("Interactive mode ready — uncomment run_interactive_pipeline() above to start.")

🏥 Medical AI Pipeline — Interactive Mode
   Describe your symptoms. Type 'quit' to exit.




💬 Your symptoms:  Patient has sharp abdominal pain, vomiting, headache, diarrhea, burning abdominal pain.


───────────────────────────────────────────────────────
  INPUT: Patient has sharp abdominal pain, vomiting, headache, diarrhea, burning abdominal pain.
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'patient has sharp abdominal pain, vomiting, headache, diarrhea and burning abdominal discomfort'
  [2] Intent Classifier: Medical  (P(medical)=1.0, threshold=0.6187)
  [3] Disease Predictor:
       1. gastritis                                     47.8%
       2. ileus                                         27.1%
       3. noninfectious gastroenteritis                 7.3%
  [4] Drug Lookup (MED-RT):
       gastritis                          : cat's claw preparation, isopropamide, isopropamide iodide, methantheline, methantheline bromide ...
Error: list indices must be integers or slices, not str



💬 Your symptoms:  Patient has shortness of breath, insomnia, breathing fast.


───────────────────────────────────────────────────────
  INPUT: Patient has shortness of breath, insomnia, breathing fast.
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'patient has short term breath, insomnia, breathing fast'
  [2] Intent Classifier: Medical  (P(medical)=0.8032, threshold=0.6187)
  [3] Disease Predictor:
       1. panic disorder                                95.5%
       2. hypertensive heart disease                    1.1%
       3. anxiety                                       0.8%
  [4] Drug Lookup (MED-RT):
       panic disorder                     : alprazolam, clomipramine, clomipramine hydrochloride, clonazepam, diazepam ...
       hypertensive heart disease         : (no results)
       anxiety                            : cannabidiol, tibolone, zolazepam, zolazepam hydrochloride
  [5] Interaction Checker: 28 interaction(s) found
       🟡 [MEDIUM] Clonazepam ⟷ Fluvoxamine
          → The metabolism of Fluvoxamine can be d

...

───────────────────────────────────────────────────────
  INPUT: Patient has anxiety and nervousness, shortness of breath, depressive or psychotic symptoms, chest tightness, palpitations, irregular heartbeat, breathing fast.
───────────────────────────────────────────────────────
  [1] Spell Corrector  : → 'patient has anxiety and nervousness, short term breath, depression or psychotic symptoms, chest tightness and palpitations, irregular heartbeat, breathing fast'
  [2] Intent Classifier: Medical  (P(medical)=0.995, threshold=0.6187)
  [3] Disease Predictor:
       1. panic disorder                                98.6%
       2. anxiety                                       1.1%
       3. acute stress reaction                         0.0%
  [4] Drug Lookup (MED-RT):
       panic disorder                     : alprazolam, clomipramine, clomipramine hydrochloride, clonazepam, diazepam ...
       anxiety                            : cannabidiol, tibolone, zolazepam, zolazepam hydrochlori


💬 Your symptoms:  patient has fever and severe cough with difficulty breathing 


───────────────────────────────────────────────────────
  INPUT: patient has fever and severe cough with difficulty breathing
───────────────────────────────────────────────────────
  [1] Spell Corrector  : (unchanged)
  [2] Intent Classifier: Medical  (P(medical)=0.9998, threshold=0.6187)
  [3] Disease Predictor:
       1. acute respiratory distress syndrome (ards)    23.7%
       2. acute bronchospasm                            18.2%
       3. asthma                                        17.0%
  [4] Drug Lookup (MED-RT):
       acute respiratory distress syndrome (ards): (no results)
       acute bronchospasm                 : (no results)
       asthma                             : albuterol, albuterol sulfate, aminophylline, aminophylline dihydrate, aminophylline hydrate ...
  [5] Interaction Checker: 0 interaction(s) found

───────────────────────────────────────────────────────
📋 MEDICAL ASSESSMENT
Input (corrected): patient has fever and severe cough with difficulty breathing




💬 Your symptoms:  exit


Goodbye! 👋
Interactive mode ready — uncomment run_interactive_pipeline() above to start.
